In [ ]:
#Подключение библиотек
import pandas as pd
import pyodbc
import numpy as np
import plotly.express as px

In [ ]:
driver = 'DRIVER={SQL Server}'
server = 'SERVER=Servtcs.'
port = 'PORT=1433'
db = 'DATABASE=TestSergey'
user = 'UID=sa'
pw = 'PWD=password'
conn_str = ';'.join([driver, server, port, db, user, pw])
pd.set_option('display.max_columns', None) #Отменяет ограничение на количество выводимых колонок
#pd.set_option('display.max_rows', None) # Отменяет ограничение на количество выводимых строк.
#pd.set_option('display.max_colwidth', None) # Отменяет ограничение на ширину ячеек (количество символов в одной ячейке).

In [3]:
SQL = '''
SET NOCOUNT ON;
DECLARE @t DATETIME =  GETDATE(); --CONVERT(DATE, '01.12.2025', 104);
DECLARE @startdate DATETIME = DATEADD(day, -90, @t);

WITH
--Распределяю на 3 месяца/группы (по времени входа), если нет даты выхода, время беру времени входа
RawSessions AS (
    SELECT 
        ls.user_id,
        ls.ls_timein AS s_start,
        ISNULL(ls.ls_timeout, ls.ls_timein) AS s_end,
        CASE 
            WHEN DATEDIFF(day, ls.ls_timein, @t) < 30 THEN 1 
            WHEN DATEDIFF(day, ls.ls_timein, @t) < 60 THEN 2 
            ELSE 3 
        END AS monthnum
    FROM log_session ls
    WHERE ls.ls_timein >= @startdate 
      AND ls.ls_timein <= @t AND ls.ls_timeout IS NOT NULL
),
--Ищу максимальное время окончания предыдущих сессий пользователя
SessionPeaks AS (
    SELECT *,
        MAX(s_end) OVER (
            PARTITION BY user_id 
            ORDER BY s_start, s_end 
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ) AS max_prev_end
    FROM RawSessions
),
--Если s_start текущей строки строго больше, чем max_prev_end, значит, пересечения нет — это старт новой чистой группы
SessionGroups AS (
    SELECT *,
        CASE 
            WHEN max_prev_end IS NULL OR s_start > max_prev_end THEN 1 
            ELSE 0 
        END AS is_new_start
    FROM SessionPeaks
),
--Формирую ID групп через нарастающий итог
SessionIslands AS (
    SELECT *,
        SUM(is_new_start) OVER (
            PARTITION BY user_id 
            ORDER BY s_start, s_end 
            ROWS UNBOUNDED PRECEDING
        ) AS group_id
    FROM SessionGroups
),
--Схлопываю пересекающиеся интервалы внутри групп (нахожу их реальные Start и End)
MergedSessions AS (
    SELECT 
        user_id,
        MIN(s_start) AS merged_start,
        MAX(s_end) AS merged_end,
        -- Переношу категорию месяца
        MIN(monthnum) AS monthnum,
        -- Для подсчета сырого кол-ва сессий (сколько их было изначально в этой группе)
        COUNT(*) AS sessions_in_group
    FROM SessionIslands
    GROUP BY user_id, group_id
),
--Считаю разницу времени уже БЕЗ наложений
PreparedData AS (
    SELECT 
        user_id,
        CAST(merged_start AS DATE) AS ls_date,
        -- Чистая разница в часах без наложений
        DATEDIFF(minute, merged_start, merged_end) / 60.0 AS hoursdiff,
        -- Количество физических сессий
        sessions_in_group AS is_session,
        monthnum
    FROM MergedSessions
),
--Агрегации
TableSessions AS (
    SELECT 
        user_id,
        -- Сумма часов без наожения по периодам
        SUM(CASE WHEN monthnum = 1 THEN hoursdiff ELSE 0 END) AS [Часы: 0-29 дней назад],
        SUM(CASE WHEN monthnum = 2 THEN hoursdiff ELSE 0 END) AS [Часы: 30-59 дней назад],
        SUM(CASE WHEN monthnum = 3 THEN hoursdiff ELSE 0 END) AS [Часы: 60-89 дней назад],
        SUM(hoursdiff) AS [Всего часов за 90 дней],
        
        -- Количество сессий по периодам
        SUM(CASE WHEN monthnum = 1 THEN is_session ELSE 0 END) AS [Сессии: 0-29 дней назад],
        SUM(CASE WHEN monthnum = 2 THEN is_session ELSE 0 END) AS [Сессии: 30-59 дней назад],
        SUM(CASE WHEN monthnum = 3 THEN is_session ELSE 0 END) AS [Сессии: 60-89 дней назад],
        SUM(is_session) AS [Всего сессий за 90 дней],

        -- Количество уникальных дней с сессиями
        COUNT(DISTINCT CASE WHEN monthnum = 1 THEN ls_date END) AS [Уникальные сессии дни: 0-29 дней назад],
        COUNT(DISTINCT CASE WHEN monthnum = 2 THEN ls_date END) AS [Уникальные сессии дни: 30-59 дней назад],
        COUNT(DISTINCT CASE WHEN monthnum = 3 THEN ls_date END) AS [Уникальные сессии дни: 60-89 дней назад],
        COUNT(DISTINCT ls_date) AS [Всего уникальных сессий за 90 дней]
    FROM PreparedData
    GROUP BY user_id
),
USERSTCS
    AS
    (
        SELECT u.USER_ID [ID работинка],
            u.USER_NAME [Пользователь] ,
            u.USER_LAST_NAME [Фамилия],
            u.USER_FIRST_NAME [Имя],
            u.USER_MIDDLE_NAME [Отчество],
            u.USER_COMMENT [Комментарий],
            u.USER_BUST [Уволен],
            u.USER_PHONE [Телефон],
            u.USER_EMAIL [EMAIL]
        FROM USERS u
    ),
    MSZ_LOG
    AS
    (
        SELECT DISTINCT
            ls.USER_ID,
            ls.LS_NAME [Компьютер],
            ls.LS_USERNAME [Пользователь ПК],
            ls.LS_TIMEIN [Последнее время входа],
            ls.LS_ID
        FROM LOG_SESSION ls
        WHERE ls.LS_TIMEIN = (SELECT MAX(LS_TIMEIN) FROM LOG_SESSION WHERE LOG_SESSION.USER_ID = ls.USER_ID)
    ), 
    MSZ_REZ
    AS
    (
        SELECT DISTINCT USERSTCS.*,
            ml.Компьютер,
            ml.[Пользователь ПК],
            ml.[Последнее время входа], 
            CASE 
                  WHEN loging.LT_COMMENT LIKE '%TCS_PLATFORM%' THEN 'Платформа'
                  WHEN loging.LT_COMMENT LIKE '%TCS_MAN%' THEN 'Производство'
                  WHEN loging.LT_COMMENT LIKE '%TCS_PDM%' THEN 'PDM'  
                  WHEN loging.LT_COMMENT LIKE '%TCS_MDM%' THEN 'MDM'  
                  WHEN loging.LT_COMMENT LIKE '%TCS_MES%' THEN 'Производственный учет'  
                  WHEN loging.LT_COMMENT LIKE '%TCS_WMS%' THEN 'Складской учет'   
                  WHEN loging.LT_COMMENT LIKE '%TCS_TDM%' THEN 'Документ'   
                  WHEN loging.LT_COMMENT LIKE '%TCS_EAM%' THEN 'EAM' 
                  WHEN loging.LT_COMMENT LIKE '%TCS_CAPP%' THEN 'CAPP' 
                  --WHEN loging.LT_COMMENT LIKE 'TCS%' THEN 'None' 
                    
                  END  AS 'Конфигурация'
        FROM USERSTCS
            LEFT JOIN MSZ_LOG ml ON USERSTCS.[ID работинка] = ml.USER_ID
            LEFT JOIN LOG_TABLE AS loging ON ml.LS_ID = loging.LS_ID
        WHERE LT_COMMENT LIKE '%TCS%' AND NOT LT_COMMENT LIKE '%TCS_RPT%' AND NOT LT_COMMENT LIKE '%TCS_API%' AND (
          loging.LT_COMMENT LIKE '%TCS_PLATFORM%' OR
            loging.LT_COMMENT LIKE '%TCS_MAN%' OR
            loging.LT_COMMENT LIKE '%TCS_PDM%' OR
            loging.LT_COMMENT LIKE '%TCS_MDM%' OR
            loging.LT_COMMENT LIKE '%TCS_MES%' OR
            loging.LT_COMMENT LIKE '%TCS_WMS%' OR
            loging.LT_COMMENT LIKE '%TCS_TDM%' OR
            loging.LT_COMMENT LIKE '%TCS_EAM%' OR
            loging.LT_COMMENT LIKE '%TCS_CAPP%'
            --loging.LT_COMMENT LIKE '%TCS%'
      )
    ),
    TableUser
    AS
    (
        SELECT MSZ_REZ.*,
            TableSessions.[Уникальные сессии дни: 0-29 дней назад],
            TableSessions.[Уникальные сессии дни: 30-59 дней назад],
            TableSessions.[Уникальные сессии дни: 60-89 дней назад],
            TableSessions.[Часы: 0-29 дней назад],
            TableSessions.[Часы: 30-59 дней назад],
            TableSessions.[Часы: 60-89 дней назад]
        FROM MSZ_REZ
            LEFT JOIN TableSessions ON MSZ_REZ.[ID работинка] = TableSessions.[USER_ID]
        WHERE MSZ_REZ.Уволен = 'F'
    ),
    HierarchyTree AS
    (
        SELECT
                u.user_id,
                c.commontree_id,
                c.commontree_parent,
                c.commontree_name,
                1 AS level
            FROM commontree c
                INNER JOIN users u ON c.commontree_id = u.commontree_id
            WHERE c.commontree_type = 30

        UNION ALL

            SELECT
                h.user_id, 
                d.commontree_id,
                d.commontree_parent,
                d.commontree_name,
                h.level + 1
            FROM commontree d
                INNER JOIN HierarchyTree h ON d.commontree_id = h.commontree_parent
            WHERE h.commontree_parent <> 22 AND d.commontree_type = 30
    ),
    RankedTree
    AS
    (
        --Нумеруем строки для каждого пользователя отдельно (самый высокий уровень получит 1)
        SELECT
            user_id,
            commontree_id,
            commontree_parent,
            commontree_name,
            ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY level DESC) AS rn
        FROM HierarchyTree
    ),
    
    Tree AS (
        SELECT
            user_id,
            commontree_id,
            commontree_parent,
            commontree_name
        FROM RankedTree
        WHERE rn = 1 
)
SELECT 
    Tree.COMMONTREE_NAME,
    TableUser.*
    FROM TableUser
    LEFT JOIN Tree ON TableUser.[ID работинка] = Tree.[USER_ID]
    ORDER BY Tree.COMMONTREE_NAME
'''

In [4]:
conn = pyodbc.connect(conn_str)
try:
    cursor = conn.cursor()
    cursor.execute(SQL)
    rows = cursor.fetchall()
    columns = [column[0] for column in cursor.description] 
finally:
    conn.close()

tcs = pd.DataFrame.from_records(rows, columns=columns)


In [5]:
#Очищаю данные от пустых значений
tcs_clean = tcs.dropna(subset=[
    'COMMONTREE_NAME', 
    'Пользователь', 
    'Уникальные сессии дни: 0-29 дней назад',
    'Часы: 0-29 дней назад'
])

#Исключаем строки, где часы равны нулю
tcs_clean = tcs_clean[tcs_clean['Часы: 0-29 дней назад'] > 0]
tcs_clean['Пользователь'] = 'Пользователь ' + (tcs_clean.groupby('Пользователь').ngroup() + 1).astype(str)

#Считываю уникальных людей в каждом подразделении и добавляю этот столбец в датафрейм
user_counts = tcs_clean.groupby('COMMONTREE_NAME')['Пользователь'].transform('nunique')
tcs_clean['Кол_во_пользователей'] = user_counts


fig = px.treemap(
    tcs_clean, 
    path=['COMMONTREE_NAME', 'Пользователь'], 
    values='Часы: 0-29 дней назад',                  
    color='Уникальные сессии дни: 0-29 дней назад', 
    color_continuous_scale='Blues', 
    title='Рейтинг активности пользователей системы TechnologiCS за 30 дней: размер плитки - Часы, цвет - количество дней',
    
    # Добавляю новые данные в custom_data: индекс 0 - Дни, индекс 1 - Кол-во пользователей
    custom_data=['Уникальные сессии дни: 0-29 дней назад', 'Кол_во_пользователей'], 
    
    #width=2800,  
    #height=1200 
)

# НАСТРОЙКА ОТОБРАЖЕНИЯ НА ПЛИТКЕ И ПРИ НАВЕДЕНИИ
# Использовать %{customdata[1]} можно только для плиток подразделений. 
# На плитках пользователей оно тоже отобразится, показывая общий размер его отдела.
fig.update_traces(
    texttemplate="<b>%{label}</b><br>Часы: %{value:.1f}<br>Дней: %{customdata[0]:.0f}<br>Активных сотрудников в отделе: %{customdata[1]:.0f}",
    textposition="top left",
    
    hovertemplate="<br>".join([
        "<b>Подразделение/пользователь:</b> %{label}",
        "<b>Часов активности:</b> %{value:.1f}",
        "<b>Дней активности:</b> %{customdata[0]:.0f}", 
        "<b>Активных сотрудников в отделе:</b> %{customdata[1]:.0f}",
        "<extra></extra>"
    ])
)

# Настройка внешнего вида и шрифтов
fig.update_layout(
    margin=dict(l=20, r=20, t=60, b=20),
    font=dict(size=16),
    coloraxis_colorbar=dict(
        title="Дней активности",
        title_font=dict(size=16),
        tickfont=dict(size=14)
    ),
    hoverlabel=dict(
        bgcolor="white",
        font_size=18,
        font_family="Arial"
    )
)

fig.show()


In [6]:
with open("Активность пользователей системы TechnologiCS за 30 дней.html", "w", encoding="utf-8") as f:
    f.write(fig.to_html(include_plotlyjs='cdn'))

In [7]:

df_bar = tcs_clean.groupby('COMMONTREE_NAME')['Пользователь'].nunique().reset_index()
df_bar.columns = ['Подразделение', 'Количество пользователей']


df_bar = df_bar.sort_values(by='Количество пользователей', ascending=True)

fig = px.bar(
    df_bar, 
    x='Количество пользователей', 
    y='Подразделение', 
    orientation='h',
    title='Рейтинг активности пользователей системы TechnologiCS за 30 дней',
    text='Количество пользователей', 
    color='Количество пользователей',
    color_continuous_scale='Blues'
)

fig.update_traces(textposition='outside')
fig.update_layout(yaxis={'categoryorder':'total ascending'}, height=800)
fig.show()


In [8]:
with open("Активность пользователей системы TechnologiCS за 30 дней.html", "a", encoding="utf-8") as f:
    f.write(fig.to_html(include_plotlyjs='cdn'))

In [9]:
pie = (tcs_clean.groupby('COMMONTREE_NAME')['Пользователь']
             .nunique()
             .reset_index()
             .rename(columns={'COMMONTREE_NAME': 'Подразделение'})[['Подразделение', 'Пользователь']])


In [10]:
import numpy as np
import plotly.graph_objects as go

# Агрегируем мелкие сектора в "Другие" (оставляем, например, топ-15, остальное объединяем)
top_n = 15
pie_sorted = pie.sort_values(by="Пользователь", ascending=False)

top_sectors = pie_sorted.head(top_n)
other_sectors = pie_sorted.iloc[top_n:]

if not other_sectors.empty:
    other_row = {
        "Подразделение": "Другие подразделения",
        "Пользователь": other_sectors["Пользователь"].sum(),
    }

    plot_data = pd.concat([top_sectors, pd.DataFrame([other_row])], ignore_index=True)
else:
    plot_data = top_sectors

labels = plot_data["Подразделение"]
values = plot_data["Пользователь"]

# Строим график
fig = go.Figure(
    data=[
        go.Pie(
            labels=labels,
            values=values,
            hole=0.4,  # Оптимальная ширина кольца
            textinfo="label+percent",
            textposition="outside",
            # Настройка скрытия подписей для слишком мелких секторов (менее 1% от площади)
            insidetextorientation="radial",
            automargin=True,
            marker=dict(line=dict(color="#FFFFFF", width=2)),
        )
    ]
)

#Настраиваем макет и легенду
fig.update_layout(
    title_text="Процентное соотношение активных пользователей системы TechnologiCS по подразделениям",
    title_x=0.5,
    showlegend=True,
    margin=dict(t=80, b=100, l=50, r=50),
)

fig.show()


In [11]:
with open("Активность пользователей системы TechnologiCS за 30 дней.html", "a", encoding="utf-8") as f:
    f.write(fig.to_html(include_plotlyjs='cdn'))